# ♻️ IA Zéro Déchet — Notebook Colab
Assistant de tri, réparation et réutilisation avec données fictives réalistes.


In [ ]:
!pip install -q pandas scikit-learn numpy


In [ ]:
import pandas as pd
from io import StringIO
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
csv_data = """objet,categorie,mots_cles,action_prioritaire,explication,reutilisation,score_ecologique
Téléphone brisé,électronique,"téléphone smartphone écran batterie chargeur",Réparer ou déposer en écocentre,"Les appareils électroniques contiennent des métaux rares et des composants polluants.",Transformer en caméra de sécurité ou lecteur audio,92
Ordinateur portable lent,électronique,"ordinateur laptop pc lent batterie clavier",Réparer ou reconditionner,"Un ordinateur peut souvent être prolongé avec une mise à jour logicielle ou un changement de pièce.",Installer Linux léger ou donner à un organisme,90
Pile usagée,dangereux,"pile batterie aa aaa lithium",Point de dépôt spécialisé,"Les piles ne doivent pas aller aux déchets ordinaires.",Aucune réutilisation recommandée,95
Pot de yogourt,plastique,"pot yogourt plastique contenant alimentaire",Recycler si accepté localement,"Certains plastiques sont recyclables selon le numéro et la municipalité.",Utiliser pour semis ou rangement de vis,65
Bouteille de verre,verre,"bouteille verre pot bocal",Recycler ou réutiliser,"Le verre se recycle bien, mais la réutilisation évite du transport et de l’énergie.",Vase, contenant sec ou décoration,78
Vieux manteau,textile,"manteau vêtement textile tissu hiver",Donner ou réparer,"Les textiles ont un fort impact environnemental; il vaut mieux prolonger leur vie.",Patchwork, doublure ou don communautaire,86
T-shirt troué,textile,"t-shirt chandail trou coton vêtement",Réparer ou transformer,"Un textile abîmé peut devenir chiffon ou matière créative.",Chiffon, sac réutilisable ou couture,82
Marc de café,organique,"marc café filtre compost",Composter,"La matière organique produit du méthane en enfouissement.",Engrais doux, désodorisant ou exfoliant maison,91
Épluchures de légumes,organique,"épluchures légumes fruits restes cuisine",Composter,"Les résidus alimentaires sont précieux pour le compost.",Bouillon maison si propres,93
Carton de livraison,papier,"carton boîte livraison emballage",Réutiliser ou recycler,"Le carton peut servir plusieurs fois avant recyclage.",Rangement, déménagement ou bricolage,83
Livre ancien,papier,"livre roman manuel papier",Donner,"Un livre a une valeur culturelle et peut circuler longtemps.",Boîte à livres, bibliothèque ou organisme,87
Meuble abîmé,encombrant,"meuble chaise table bois armoire",Réparer ou donner,"Les encombrants peuvent souvent être restaurés.",Upcycling, peinture, transformation,84
Médicament expiré,dangereux,"médicament pilule sirop pharmacie",Retour en pharmacie,"Les médicaments ne doivent pas être jetés aux toilettes ni aux déchets.",Aucune réutilisation,98
Jouet en bon état,réutilisable,"jouet enfant jeu peluche",Donner,"Un objet encore fonctionnel devrait être transmis.",Don à une famille, école ou organisme,88
"""

df = pd.read_csv(StringIO(csv_data))
df.head()


In [ ]:
def build_search_text(row):
    return f"{row['objet']} {row['categorie']} {row['mots_cles']} {row['action_prioritaire']}"

df['texte_recherche'] = df.apply(build_search_text, axis=1)
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
matrix = vectorizer.fit_transform(df['texte_recherche'])

def recommander(description, top_k=3):
    query_vec = vectorizer.transform([description])
    similarities = cosine_similarity(query_vec, matrix).flatten()
    top_indices = similarities.argsort()[::-1][:top_k]
    cols = ['objet','categorie','action_prioritaire','explication','reutilisation','score_ecologique']
    result = df.iloc[top_indices][cols].copy()
    result['similarite'] = [round(float(similarities[i]), 3) for i in top_indices]
    return result


In [ ]:
recommander("vieux téléphone avec écran brisé")


In [ ]:
recommander("manteau troué mais encore chaud")


## Pour aller plus loin
- Ajouter les règles de tri de votre municipalité.
- Ajouter une carte des écocentres.
- Ajouter une reconnaissance d’image.
- Déployer avec Streamlit Community Cloud ou Hugging Face Spaces.
